In [ ]:
"""First Question starts here"""

''

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [ ]:
import google.generativeai as genai
import json
import time


genai.configure(api_key="<Googel Gemini API Key>")  
MODEL_NAME = "models/gemini-2.5-pro-exp-03-25"  

def validate_mcq(mcq):
    return (
        isinstance(mcq, dict)
        and "question" in mcq
        and "options" in mcq
        and "answer" in mcq
        and isinstance(mcq["options"], list)
        and len(mcq["options"]) == 4
        and mcq["answer"] in mcq["options"]
    )

def parse_mcqs_from_text(text):
    try:
        start = text.find('{')
        end = text.rfind('}') + 1
        json_str = text[start:end]
        data = json.loads(json_str)
        return data.get("questions", [])
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        return []

def generate_mcqs():
    prompt = (
        "Generate 10 unique multiple-choice questions (MCQs) for Grade 9 on the topic 'Laws of Motion'. "
        "Each question must have exactly 4 options and one correct answer. "
        "Return the result in the following JSON format:\n"
        "{\n"
        "  \"questions\": [\n"
        "    {\n"
        "      \"question\": \"...\",\n"
        "      \"options\": [\"...\", \"...\", \"...\", \"...\"],\n"
        "      \"answer\": \"...\"\n"
        "    },\n"
        "    ...\n"
        "  ]\n"
        "}\n"
    )

    model = genai.GenerativeModel(MODEL_NAME)
    retries = 3
    for attempt in range(1, retries + 1):
        try:
            response = model.generate_content(prompt)
            text = response.text
            mcqs = parse_mcqs_from_text(text)
            valid_mcqs = [q for q in mcqs if validate_mcq(q)]

            if len(valid_mcqs) >= 10:
                with open("questions.json", "w") as f:
                    json.dump(valid_mcqs[:10], f, indent=2)
                print(f"Success: Generated and saved 10 valid MCQs on attempt {attempt}")
                return valid_mcqs[:10]
            else:
                print(f"Attempt {attempt}: Only {len(valid_mcqs)} valid MCQs found, retrying...")
                time.sleep(1)
        except Exception as e:
            print(f"Attempt {attempt}: Error during generation: {e}")
            time.sleep(1)

    print("Failed to generate 10 valid MCQs after retries.")
    return []


In [ ]:
generate_mcqs()

In [ ]:
"""First Question ends here"""

In [ ]:
"""Second Question starts here"""

In [7]:
import language_tool_python


def check_duplicate_options(options):
    """Returns a list of duplicate options, if any."""
    seen = set()
    duplicates = set()
    for opt in options:
        if opt in seen:
            duplicates.add(opt)
        else:
            seen.add(opt)
    return list(duplicates)

def check_answer_mismatch(options, answer):
    """Returns True if the answer is not in options."""
    return answer not in options

def check_grammar(text, tool=None):
    """Returns a list of grammar issues and suggested corrections."""
    if tool is None:
        tool = language_tool_python.LanguageTool('en-US')
    matches = tool.check(text)
    issues = []
    for match in matches:
        issues.append({
            "error": text[match.offset: match.offset + match.errorLength],
            "message": match.message,
            "suggestions": match.replacements
        })
    return issues

def check_mcqs(mcq_list):
    """Checks a list of MCQs for duplicate options, answer mismatches, and poor grammar."""
    tool = language_tool_python.LanguageTool('en-US')
    results = []
    for idx, mcq in enumerate(mcq_list):
        errors = []
        # 1. Check duplicate options
        duplicates = check_duplicate_options(mcq["options"])
        if duplicates:
            errors.append({
                "type": "duplicate_options",
                "message": f"Duplicate options found: {duplicates}",
                "suggestion": "Make all options unique."
            })
        # 2. Check answer mismatch
        if check_answer_mismatch(mcq["options"], mcq["answer"]):
            errors.append({
                "type": "answer_mismatch",
                "message": f"Answer '{mcq['answer']}' is not in options.",
                "suggestion": "Ensure the answer is exactly one of the options."
            })
        # 3. Check grammar
        grammar_issues = check_grammar(mcq["question"], tool)
        if grammar_issues:
            for issue in grammar_issues:
                errors.append({
                    "type": "grammar",
                    "message": f"Grammar issue: {issue['message']} (error: '{issue['error']}')",
                    "suggestion": f"Suggestions: {issue['suggestions']}"
                })
        # Suggest correction for grammar if issues found
        if grammar_issues:
            corrected = tool.correct(mcq["question"])
        else:
            corrected = None
        results.append({
            "index": idx,
            "question": mcq["question"],
            "errors": errors,
            "grammar_corrected_question": corrected if grammar_issues else None
        })
    return results

In [9]:
import pprint

mcqs = [
        {
            "question": "What is Newton's first law called?",
            "options": ["Law of inertia", "Law of inertia", "Law of force", "Law of action"],
            "answer": "Law of inertia"
        },
        {
            "question": "The force is equal to mass times acceleration.",
            "options": ["Newton's first law", "Newton's second law", "Newton's third law", "Law of gravity"],
            "answer": "Newton's law of motion"
        },
        {
            "question": "Which law state that every action have equal and opposite reaction?",
            "options": ["First law", "Second law", "Third law", "Law of inertia"],
            "answer": "Third law"
        }
    ]
results = check_mcqs(mcqs)
pprint.pprint(results)

In [ ]:
"""Second Question ends here"""